# Anomaly Detection on Well Log Data
## Machine learning for energy applications — Module 5, Exercise: Anomaly Detection

In this exercise we apply unsupervised anomaly detection techniques to well log data. Measurement anomalies in well logs can arise from tool malfunctions, borehole washouts, invasion effects, or genuinely unusual geological formations. Identifying these anomalies is a critical quality-control step before any petrophysical interpretation or machine-learning workflow.

### Learning Objectives
1. Detect unusual measurements in well log data using statistical rules (Z-score, IQR), after recording what data cleaning removed.
2. Apply Isolation Forest, Local Outlier Factor, and One-Class SVM to multivariate well log data, with one score convention: higher = more unusual.
3. Compare the methods and turn their scores into a ranked review list, knowing that agreement between methods is not validation.
4. Visualize flagged intervals on standard well log plots.

**Labs:** Lab 1 covers Parts 1–3. Lab 2 covers Parts 4–6 and ends with a written alert policy.

---
## Overview

Well log data consists of continuous measurements recorded as a function of depth (measured depth, MD). Each curve — gamma ray (GR), bulk density (RHOB), neutron porosity (NPHI), sonic transit time (DT), caliper (CALI), etc. — responds to different rock and fluid properties.

Anomalies in this data can be caused by:
- **Tool malfunctions** — spikes or drop-outs in a single curve.
- **Borehole conditions** — washouts (CALI >> BS) leading to poor pad contact for density or neutron tools.
- **Invasion effects** — drilling fluid infiltrating the formation and altering shallow resistivity readings.
- **Genuine geological anomalies** — thin beds, fractures, or mineralogical changes that produce unusual log responses.

We will explore several anomaly detection methods, from simple univariate statistics to sophisticated machine-learning algorithms, and compare their results.

### How this lab works (read before you start)

- **Protocol: an exploratory review of one fixed table.** Every rule and detector is fitted on the same complete rows that it then flags (scikit-learn's *outlier detection* mode), and the scaler is fitted on those same rows. That is legitimate for reviewing a historical data set; it is not a test of how a detector would perform on a new well. To score new wells you would fit the scaler and the detector on reference wells only (a pipeline does both), choose and freeze the threshold on reference or validation data, and use `LocalOutlierFactor(novelty=True)` for LOF.
- **No labels.** Nothing in this file says which rows are real problems. We can count flags and rank rows for review, but we cannot report precision, recall or accuracy for any method, and agreement between methods is not evidence that they are right.
- **A score is not an alert.** Each detector produces a continuous score; a flag is a separate decision (score above a threshold). In this notebook every score is oriented so that **higher = more unusual**; scikit-learn's `score_samples` uses the opposite sign.
- **`contamination` (and `nu`) is an alert budget that you choose**, not a prevalence the model discovers: `contamination=0.05` flags 5% of the rows because we asked for 5%.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

print("Libraries loaded successfully.")

---
## Part 1: Data Loading and Exploration

We load the well log dataset and perform an initial exploration — understanding the shape, columns, value ranges, and any missing data — and we record exactly what the cleaning step removes.

In [ ]:
# Load the well log data
df = pd.read_csv("well_logs_data.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nWells: {list(df['wellName'].unique())}")
print(f"Number of wells: {df['wellName'].nunique()}")
print()
display(df.head(10))
print()
display(df.describe())
print()
df.info()

In [ ]:
# Visualize key well log curves vs depth for the first well
well_name = df["wellName"].unique()[0]
well_df = df[df["wellName"] == well_name].copy()

log_curves = ["GR", "RHOB", "NPHI", "DT"]
colors = ["green", "red", "blue", "purple"]

fig, axes = plt.subplots(1, len(log_curves), figsize=(14, 10), sharey=True)
fig.suptitle(f"Well Log Display — {well_name}", fontsize=14, y=1.01)

for i, (curve, color) in enumerate(zip(log_curves, colors)):
    axes[i].plot(well_df[curve], well_df["MD"], color=color, linewidth=0.6)
    axes[i].set_xlabel(curve, fontsize=11)
    axes[i].set_title(curve, fontsize=12)
    axes[i].grid(True, alpha=0.3)

axes[0].set_ylabel("Measured Depth (MD)", fontsize=11)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Check for missing values and handle them
print("=== Missing Values per Column ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values found.")
print(f"\nTotal missing values: {df.isnull().sum().sum()}")
print(f"Total rows: {len(df)}")

# Drop rows with missing values in the key log curves
key_columns = ["GR", "RHOB", "NPHI", "DT", "CALI", "BS"]
df_clean = df.dropna(subset=key_columns).copy()
print(f"\nRows after dropping NaN in key columns: {len(df_clean)} (dropped {len(df) - len(df_clean)} rows)")

# Audit: what did the cleaning remove, well by well?
audit = pd.DataFrame({
    "rows_before": df.groupby("wellName").size(),
    "rows_after": df_clean.groupby("wellName").size(),
}).fillna(0).astype(int)
missing_by_well = df[key_columns].isna().groupby(df["wellName"]).mean()   # share of rows missing each curve
display(audit.join(missing_by_well.add_prefix("missing_").round(2)))

lost_wells = audit.index[audit["rows_after"] == 0].tolist()
print(f"Wells before cleaning: {df['wellName'].nunique()}, after: {df_clean['wellName'].nunique()}; "
      f"wells lost entirely: {lost_wells}")
print(f"Rows kept: {len(df_clean):,} of {len(df):,} ({len(df_clean) / len(df):.1%})")

**What the cleaning step costs.** `dropna` keeps a row only if all six curves are present, and it gives no warning. DT is missing in every row of two wells, **15/9-F-1 C** and **15/9-F-11 B**, so both wells disappear completely: 5,493 of the 27,845 rows (about 20%) remain, from three wells. Everything below describes those three wells, not "the field". The lab keeps the six curves so that everyone works on the same table; alternatives (drop DT and keep all five wells, or analyse each well separately) are discussed in the debrief. Write the row and well counts before and after cleaning into your notes (Lab 1 deliverable).

---
## Part 2: Statistical Anomaly Detection

We begin with classical statistical approaches:
- **Z-score method** — flags data points that fall more than 3 standard deviations from the mean in any feature.
- **IQR method** — uses the interquartile range to define fences beyond which points are considered outliers.

These methods are simple, interpretable, and computationally inexpensive, and they are the baseline that every learned detector should be compared with. Two cautions from the lecture: the mean and standard deviation are pulled by the outliers themselves (masking; a robust z-score uses the median and MAD instead), and fences computed on a mixture of rock types flag the minority rock type as a whole.

The scaler is fitted on the complete rows: the same table that every rule and detector in this notebook describes (see *How this lab works*).

In [ ]:
# Select key features and standardize
# (the scaler is fitted on the complete rows: the same table every rule and detector below describes)
features = ["GR", "RHOB", "NPHI", "DT", "CALI", "BS"]
X = df_clean[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create a DataFrame of scaled values for convenience
df_scaled = pd.DataFrame(X_scaled, columns=features, index=df_clean.index)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"\nScaled feature statistics:")
display(df_scaled.describe().round(3))

In [ ]:
# Z-score method: flag points where |z| > 3 for any feature
z_threshold = 3
z_flag_mask = (np.abs(df_scaled) > z_threshold).any(axis=1)

df_clean["zscore_flag"] = z_flag_mask.astype(int)   # 1 = flagged, 0 = not flagged

n_zscore = z_flag_mask.sum()
print(f"Z-score flags (|z| > {z_threshold}): {n_zscore} rows ({100*n_zscore/len(df_clean):.2f}%)")
print(f"\nFlagged rows per feature (|z| > {z_threshold}):")
for col in features:
    count = (np.abs(df_scaled[col]) > z_threshold).sum()
    print(f"  {col}: {count}")

In [ ]:
# Visualize z-score flags on well log plots
well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

fig, axes = plt.subplots(1, 4, figsize=(16, 10), sharey=True)
fig.suptitle(f"Z-Score Flags — {well_name}", fontsize=14, y=1.01)

for i, (curve, color) in enumerate(zip(["GR", "RHOB", "NPHI", "DT"], ["green", "red", "blue", "purple"])):
    # Plot all data
    axes[i].plot(wdf[curve], wdf["MD"], color=color, linewidth=0.5, alpha=0.7)
    # Highlight flagged rows
    flagged = wdf[wdf["zscore_flag"] == 1]
    axes[i].scatter(flagged[curve], flagged["MD"], color="red", s=8, zorder=5, label="Flagged")
    axes[i].set_xlabel(curve, fontsize=11)
    axes[i].set_title(curve, fontsize=12)
    axes[i].grid(True, alpha=0.3)

axes[0].set_ylabel("Measured Depth (MD)", fontsize=11)
axes[0].invert_yaxis()
axes[-1].legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# IQR method: flag points outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
Q1 = df_clean[features].quantile(0.25)
Q3 = df_clean[features].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outside_fences = (df_clean[features] < lower_bound) | (df_clean[features] > upper_bound)   # one column per feature
iqr_flag_mask = outside_fences.any(axis=1)
df_clean["iqr_flag"] = iqr_flag_mask.astype(int)   # 1 = flagged, 0 = not flagged

n_iqr = iqr_flag_mask.sum()
print(f"IQR flags: {n_iqr} rows ({100*n_iqr/len(df_clean):.2f}%)")
print(f"\nIQR bounds and flagged rows per feature:")
for col in features:
    print(f"  {col}: [{lower_bound[col]:.4f}, {upper_bound[col]:.4f}]  flagged: {outside_fences[col].sum()}")

In [ ]:
# Compare Z-score and IQR detections
both = (df_clean["zscore_flag"] == 1) & (df_clean["iqr_flag"] == 1)
only_zscore = (df_clean["zscore_flag"] == 1) & (df_clean["iqr_flag"] == 0)
only_iqr = (df_clean["zscore_flag"] == 0) & (df_clean["iqr_flag"] == 1)

print("=== Statistical Method Comparison ===")
print(f"  Z-score only:  {only_zscore.sum()}")
print(f"  IQR only:      {only_iqr.sum()}")
print(f"  Both methods:  {both.sum()}")
print(f"  Either method: {(df_clean['zscore_flag'] | df_clean['iqr_flag']).sum()}")

# Venn-style bar chart
fig, ax = plt.subplots(figsize=(6, 4))
categories = ["Z-score only", "IQR only", "Both"]
counts = [only_zscore.sum(), only_iqr.sum(), both.sum()]
bars = ax.bar(categories, counts, color=["steelblue", "coral", "mediumpurple"], edgecolor="k")
ax.set_ylabel("Number of Flagged Rows")
ax.set_title("Statistical Rules: Flagged Rows Compared")
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(count),
            ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Review one flagged interval (Lab 1 deliverable)
# 1) Which curve drives the IQR flags, and what kind of rock are those rows?
gr_flags = outside_fences["GR"]
print(f"IQR flags from GR: {gr_flags.sum()} rows, "
      f"{(df_clean.loc[gr_flags, 'GR'] > upper_bound['GR']).sum()} of them above the upper fence")
print(f"Median VSH (shale volume) of the GR-flagged rows: {df_clean.loc[gr_flags, 'VSH'].median():.2f}; "
      f"all complete rows: {df_clean['VSH'].median():.2f}")

# 2) Group consecutive IQR-flagged rows of the plotted well into depth intervals
well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name]
flag = wdf["iqr_flag"]
run_id = (flag != flag.shift()).cumsum().rename("interval")   # new id whenever the flag switches on or off
intervals = (wdf[flag == 1]
             .groupby(run_id[flag == 1])
             .agg(top_MD=("MD", "min"), base_MD=("MD", "max"), rows=("MD", "size"),
                  GR=("GR", "median"), RHOB=("RHOB", "median"), NPHI=("NPHI", "median"),
                  CALI=("CALI", "median"), VSH=("VSH", "median"))
             .sort_values("rows", ascending=False))
print(f"\n{well_name}: {len(intervals)} flagged intervals; the five thickest (medians of each curve):")
display(intervals.head(5).round(2))

# 3) Which curves are outside their fences in the thickest interval?
top_rows = wdf.index[(run_id == intervals.index[0]) & (flag == 1)]
print("Rows outside the fences in the thickest interval, per curve:")
print(outside_fences.loc[top_rows].sum().to_string())

**Reading the review.** All 398 GR flags lie above the upper fence, and those rows have a median VSH of 0.79 against 0.28 for all complete rows. The thickest flagged interval in 15/9-F-1 A (3620.8–3627.5 m MD, 68 rows) is outside the fences on GR alone, with a median GR of about 113 API and a median VSH of 0.83: a shale-rich interval that is unusual in a sand-dominated sample, not a bad measurement. The IQR fences flag the minority rock type as a whole (the mixture effect). VSH is a shale-volume estimate supplied in the file; it is derived from the logs, so it supports this judgement but is not an independent label.

Note also BS (bit size): it is 8.5 in in every remaining row, so its z-score is 0 everywhere and its fences collapse to [8.5, 8.5]. A constant feature contributes nothing to any rule or detector; remove such features explicitly in your own work.

---
## Part 3: Isolation Forest

**Isolation Forest** works on a fundamentally different principle: anomalies are *easier to isolate* than normal points. The algorithm builds an ensemble of random trees that recursively split the feature space. Anomalous points tend to be isolated in fewer splits (shorter path lengths).

Key parameters:
- `contamination` — the share of rows to flag: an **alert budget** you choose (for example from the number of intervals a reviewer can check), not the proportion of anomalies the model discovers. It only sets the threshold on the score; it does not change the ranking.
- `n_estimators` — number of isolation trees.
- `random_state` — for reproducibility.

**Score orientation.** In scikit-learn, `score_samples` is *higher for more normal* rows, `decision_function` is `score_samples` minus the threshold `offset_` (set by `contamination`), and `predict` / `fit_predict` return −1 for flagged rows and +1 for the others. The helper below converts every detector in this notebook, once, to **higher = more unusual**, and a check tests that convention against the flags. For Isolation Forest the converted score is the anomaly score *s* of the original paper (near 1: isolated quickly; well below 0.5: ordinary).

In [ ]:
# One score convention for the whole notebook: HIGHER = MORE UNUSUAL.
def unusualness(detector, X=None):
    """Anomaly score of a fitted scikit-learn detector, oriented so that higher = more unusual."""
    if isinstance(detector, LocalOutlierFactor) and not detector.novelty:
        return -detector.negative_outlier_factor_   # outlier mode: scores of the rows it was fitted on
    return -detector.score_samples(X)               # score_samples is higher for more normal rows


def check_orientation(score, flagged):
    """Test the convention: every flagged row must score higher than every unflagged row."""
    score, flagged = np.asarray(score), np.asarray(flagged, dtype=bool)
    assert score[flagged].min() > score[~flagged].max(), "score orientation does not match the flags"
    print(f"Orientation check passed: all {flagged.sum()} flagged rows score higher than the other {(~flagged).sum()}.")

In [ ]:
# Fit Isolation Forest (outlier mode: fitted on the rows it flags)
iso_forest = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
iso_preds = iso_forest.fit_predict(X_scaled)

# fit_predict labels: -1 = flagged (outlier), +1 = not flagged (inlier)
print(f"fit_predict labels: -1 (flagged) = {(iso_preds == -1).sum()}, +1 (not flagged) = {(iso_preds == 1).sum()}")
df_clean["iso_forest_flag"] = (iso_preds == -1).astype(int)   # stored as 1 = flagged, 0 = not flagged

n_iso = (iso_preds == -1).sum()
print(f"Isolation Forest flags (contamination=0.05): {n_iso} rows ({100*n_iso/len(df_clean):.2f}%)")

# Score: higher = more unusual. The threshold on the same scale is set by contamination.
df_clean["iso_forest_score"] = unusualness(iso_forest, X_scaled)
iso_threshold = -iso_forest.offset_
check_orientation(df_clean["iso_forest_score"], iso_preds == -1)
print(f"\nScore range: [{df_clean['iso_forest_score'].min():.4f}, {df_clean['iso_forest_score'].max():.4f}]; "
      f"flagged when score > {iso_threshold:.4f}")

In [ ]:
# Visualize Isolation Forest flags on well log plots
well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

fig, axes = plt.subplots(1, 5, figsize=(20, 10), sharey=True)
fig.suptitle(f"Isolation Forest Flags — {well_name}", fontsize=14, y=1.01)

curves = ["GR", "RHOB", "NPHI", "DT"]
colors = ["green", "red", "blue", "purple"]

for i, (curve, color) in enumerate(zip(curves, colors)):
    axes[i].plot(wdf[curve], wdf["MD"], color=color, linewidth=0.5, alpha=0.7)
    flagged = wdf[wdf["iso_forest_flag"] == 1]
    axes[i].scatter(flagged[curve], flagged["MD"], color="red", s=8, zorder=5, label="Flagged")
    axes[i].set_xlabel(curve, fontsize=11)
    axes[i].set_title(curve, fontsize=12)
    axes[i].grid(True, alpha=0.3)

# Score track (higher = more unusual), shaded where the score is above the threshold
axes[4].plot(wdf["iso_forest_score"], wdf["MD"], color="black", linewidth=0.5)
axes[4].axvline(x=iso_threshold, color="red", linestyle="--", linewidth=1, label="Threshold")
axes[4].fill_betweenx(wdf["MD"], iso_threshold, wdf["iso_forest_score"],
                      where=wdf["iso_forest_score"] > iso_threshold, color="red", alpha=0.3)
axes[4].set_xlabel("Score (higher = more unusual)", fontsize=11)
axes[4].set_title("IF Score", fontsize=12)
axes[4].grid(True, alpha=0.3)
axes[4].legend(loc="lower right")

axes[0].set_ylabel("Measured Depth (MD)", fontsize=11)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Sweep the alert budget: contamination = 0.01, 0.05, 0.10
contamination_values = [0.01, 0.05, 0.1]
results = {}
flagged_sets = {}

for contam in contamination_values:
    model = IsolationForest(contamination=contam, n_estimators=100, random_state=42)
    preds = model.fit_predict(X_scaled)
    n_flag = (preds == -1).sum()
    results[contam] = n_flag
    flagged_sets[contam] = set(df_clean.index[preds == -1])
    print(f"contamination={contam:.2f}: {n_flag} flagged rows ({100*n_flag/len(df_clean):.2f}%), "
          f"flagged when score > {-model.offset_:.4f}")

# Same random_state: the same forest and the same ranking. Only the threshold moves, so the sets are nested.
for small, large in zip(contamination_values[:-1], contamination_values[1:]):
    print(f"Every row flagged at {small:.2f} is also flagged at {large:.2f}: "
          f"{flagged_sets[small] <= flagged_sets[large]}")

# Bar chart of flagged-row counts
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([str(c) for c in contamination_values], list(results.values()),
       color=["lightblue", "steelblue", "darkblue"], edgecolor="k")
ax.set_xlabel("Contamination (alert budget)")
ax.set_ylabel("Number of Flagged Rows")
ax.set_title("Isolation Forest: Effect of Contamination Parameter")
plt.tight_layout()
plt.show()

**What contamination controls (Lab 1 deliverable).** The counts are 1%, 5% and 10% of the rows because that is what we asked for. The forest, and therefore every row's score and the ranking, is the same in all three fits (same `random_state`); contamination only moves the threshold down that one ranking, so the flagged sets are nested. It is our assumption, an alert budget, not a finding about how many rows are bad. Choosing it needs evidence from outside the model: reviewed intervals, or the number of intervals a petrophysicist can review. When a detector is used on new wells, the threshold is chosen on reference or validation data and frozen before it is used.

---
## Part 4: Local Outlier Factor

**Local Outlier Factor (LOF)** measures the local density deviation of a data point with respect to its neighbours. Points in sparser regions (relative to their neighbours) receive higher LOF scores and are more likely to be flagged as anomalies.

Key parameters:
- `n_neighbors` — the number of neighbours used to compute local density; it decides what counts as "local".
- `contamination` — again an alert budget: LOF flags that share of the rows whatever `n_neighbors` is, so compare *which* rows are flagged, not how many.

LOF is used here in outlier mode: `fit_predict` on the rows being described, with the scores in `negative_outlier_factor_`. To score new wells you would fit `LocalOutlierFactor(novelty=True)` on reference data and call `predict` or `score_samples` on the new rows only. The `unusualness` helper returns the LOF value itself: about 1 for a row as dense as its neighbours, well above 1 for a sparser row.

In [ ]:
# Fit Local Outlier Factor (outlier mode: fit_predict on the rows it flags)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_preds = lof.fit_predict(X_scaled)             # -1 = flagged, +1 = not flagged

df_clean["lof_flag"] = (lof_preds == -1).astype(int)
df_clean["lof_score"] = unusualness(lof)           # the LOF value: higher = more unusual
lof_threshold = -lof.offset_
check_orientation(df_clean["lof_score"], lof_preds == -1)

n_lof = (lof_preds == -1).sum()
print(f"LOF flags (n_neighbors=20, contamination=0.05): {n_lof} rows ({100*n_lof/len(df_clean):.2f}%)")
print(f"\nLOF score range: [{df_clean['lof_score'].min():.4f}, {df_clean['lof_score'].max():.4f}]; "
      f"flagged when score > {lof_threshold:.4f}")

In [ ]:
# Visualize LOF flags on well log plots
well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

fig, axes = plt.subplots(1, 5, figsize=(20, 10), sharey=True)
fig.suptitle(f"Local Outlier Factor Flags — {well_name}", fontsize=14, y=1.01)

curves = ["GR", "RHOB", "NPHI", "DT"]
colors = ["green", "red", "blue", "purple"]

for i, (curve, color) in enumerate(zip(curves, colors)):
    axes[i].plot(wdf[curve], wdf["MD"], color=color, linewidth=0.5, alpha=0.7)
    flagged = wdf[wdf["lof_flag"] == 1]
    axes[i].scatter(flagged[curve], flagged["MD"], color="orange", s=8, zorder=5, label="Flagged")
    axes[i].set_xlabel(curve, fontsize=11)
    axes[i].set_title(curve, fontsize=12)
    axes[i].grid(True, alpha=0.3)

# LOF score track (higher = more unusual), shaded where the score is above the threshold
axes[4].plot(wdf["lof_score"], wdf["MD"], color="black", linewidth=0.5)
axes[4].axvline(x=lof_threshold, color="orange", linestyle="--", linewidth=1, label="Threshold")
axes[4].fill_betweenx(wdf["MD"], lof_threshold, wdf["lof_score"],
                      where=wdf["lof_score"] > lof_threshold, color="orange", alpha=0.3)
axes[4].set_xlabel("LOF (higher = more unusual)", fontsize=11)
axes[4].set_title("LOF Score", fontsize=12)
axes[4].grid(True, alpha=0.3)
axes[4].legend(loc="lower right")

axes[0].set_ylabel("Measured Depth (MD)", fontsize=11)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Sweep n_neighbors with the budget fixed at contamination=0.05: compare flagged sets, not counts
neighbor_values = [5, 10, 20, 50, 100]
lof_sets = {}

for n in neighbor_values:
    model = LocalOutlierFactor(n_neighbors=n, contamination=0.05)
    preds = model.fit_predict(X_scaled)
    lof_sets[n] = set(df_clean.index[preds == -1])
    print(f"n_neighbors={n:3d}: {len(lof_sets[n])} flagged rows ({100*len(lof_sets[n])/len(df_clean):.2f}%)")


def jaccard(a, b):
    """Size of the intersection divided by the size of the union: 1 = the same rows, 0 = none in common."""
    return len(a & b) / len(a | b)


jaccard_matrix = pd.DataFrame([[jaccard(lof_sets[a], lof_sets[b]) for b in neighbor_values]
                               for a in neighbor_values], index=neighbor_values, columns=neighbor_values)
print("\nJaccard overlap of the flagged sets:")
display(jaccard_matrix.round(2))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(jaccard_matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set_xlabel("n_neighbors")
ax.set_ylabel("n_neighbors")
ax.set_title("LOF: same count, different rows (Jaccard overlap)")
plt.tight_layout()
plt.show()

---
## Part 5: One-Class SVM

**One-Class SVM** learns a decision boundary that encompasses the majority of the training data. Points falling outside the boundary are classified as anomalies. It uses a kernel (typically RBF) to map data into a higher-dimensional space where the boundary is easier to construct.

Key parameters:
- `nu` — an upper bound on the share of training rows left outside the boundary (and a lower bound on the share of support vectors). Like `contamination`, it is a budget you choose, not an estimate of how many rows are faulty. The share actually flagged is close to `nu` but not exactly equal to it, because many rows sit almost exactly on the boundary.
- `kernel` — the kernel function (`rbf` is the default and most common choice).
- `gamma` — kernel coefficient: how tightly the boundary wraps the data. `"scale"` (the default) uses 1 / (number of features × variance of X).

Training time grows quickly with the number of rows, so the code fits on a random subsample above 10,000 rows (this table is smaller). The scaler and the SVM sit in one pipeline, so the scaler is always fitted on exactly the rows the SVM is fitted on. One-Class SVM scores have an arbitrary scale (after the sign flip they are negative here); only their order matters, which is why Part 6 compares ranks, not raw scores.

In [ ]:
# Fit One-Class SVM
# Training time grows quickly with the number of rows: above 10,000 rows, fit on a random subsample.
rng = np.random.default_rng(42)
if len(X) > 10000:
    print(f"Dataset has {len(X)} samples. Subsampling to 10000 for One-Class SVM training.")
    fit_rows = rng.choice(len(X), size=10000, replace=False)
else:
    fit_rows = np.arange(len(X))

# Scaler and SVM in one pipeline: the scaler is fitted on the same rows as the SVM
ocsvm = make_pipeline(StandardScaler(), OneClassSVM(nu=0.05, kernel="rbf", gamma="scale"))
ocsvm.fit(X[fit_rows])

# Flag and score every row of the table (predict: -1 = flagged, +1 = not flagged)
svm_preds = ocsvm.predict(X)
df_clean["ocsvm_flag"] = (svm_preds == -1).astype(int)
df_clean["ocsvm_score"] = unusualness(ocsvm, X)   # higher = more unusual (arbitrary scale)
ocsvm_threshold = -ocsvm[-1].offset_[0]         # offset_ is stored as a one-element array
check_orientation(df_clean["ocsvm_score"], svm_preds == -1)

n_svm = (svm_preds == -1).sum()
print(f"One-Class SVM flags (nu=0.05): {n_svm} rows ({100*n_svm/len(df_clean):.2f}%)")
print(f"\nScore range: [{df_clean['ocsvm_score'].min():.4f}, {df_clean['ocsvm_score'].max():.4f}]; "
      f"flagged when score > {ocsvm_threshold:.4f}")

In [ ]:
# Visualize One-Class SVM flags
well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

fig, axes = plt.subplots(1, 5, figsize=(20, 10), sharey=True)
fig.suptitle(f"One-Class SVM Flags — {well_name}", fontsize=14, y=1.01)

curves = ["GR", "RHOB", "NPHI", "DT"]
colors = ["green", "red", "blue", "purple"]

for i, (curve, color) in enumerate(zip(curves, colors)):
    axes[i].plot(wdf[curve], wdf["MD"], color=color, linewidth=0.5, alpha=0.7)
    flagged = wdf[wdf["ocsvm_flag"] == 1]
    axes[i].scatter(flagged[curve], flagged["MD"], color="magenta", s=8, zorder=5, label="Flagged")
    axes[i].set_xlabel(curve, fontsize=11)
    axes[i].set_title(curve, fontsize=12)
    axes[i].grid(True, alpha=0.3)

# SVM score track (higher = more unusual), shaded where the score is above the threshold
axes[4].plot(wdf["ocsvm_score"], wdf["MD"], color="black", linewidth=0.5)
axes[4].axvline(x=ocsvm_threshold, color="magenta", linestyle="--", linewidth=1, label="Threshold")
axes[4].fill_betweenx(wdf["MD"], ocsvm_threshold, wdf["ocsvm_score"],
                      where=wdf["ocsvm_score"] > ocsvm_threshold, color="magenta", alpha=0.3)
axes[4].set_xlabel("Score (higher = more unusual)", fontsize=11)
axes[4].set_title("OCSVM Score", fontsize=12)
axes[4].grid(True, alpha=0.3)
axes[4].legend(loc="lower right")

axes[0].set_ylabel("Measured Depth (MD)", fontsize=11)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

---
## Part 6: Method Comparison

Each anomaly detection method has different strengths:
- **Z-score / IQR** — fast and interpretable but only considers marginal distributions (univariate).
- **Isolation Forest** — efficient, handles multivariate data well, good for high-dimensional datasets.
- **LOF** — captures local density variations, good for datasets with clusters of different densities.
- **One-Class SVM** — effective when the normal data occupies a compact region in feature space.

We now compare how many rows each method flags and the overlap between methods, count votes, and build a **review order** from the scores. There are no labels: agreement between methods tells you which rows to review first, not which rows are wrong.

In [ ]:
# Compare all methods — count of flagged rows and overlap
methods = ["zscore_flag", "iqr_flag", "iso_forest_flag", "lof_flag", "ocsvm_flag"]
method_names = ["Z-Score", "IQR", "Isolation Forest", "LOF", "One-Class SVM"]

print("=== Flagged Rows by Method ===")
counts_dict = {}
for name, col in zip(method_names, methods):
    n = df_clean[col].sum()
    pct = 100 * n / len(df_clean)
    counts_dict[name] = n
    print(f"  {name:20s}: {n:6d} ({pct:.2f}%)")

# Pairwise overlap matrix: rows flagged by both methods (the diagonal is each method's count)
print("\n=== Pairwise Overlap (number of rows flagged by both methods) ===")
overlap_matrix = pd.DataFrame(0, index=method_names, columns=method_names)
for name_i, col_i in zip(method_names, methods):
    for name_j, col_j in zip(method_names, methods):
        overlap_matrix.loc[name_i, name_j] = ((df_clean[col_i] == 1) & (df_clean[col_j] == 1)).sum()

display(overlap_matrix)

# Heatmap of overlap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(overlap_matrix, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
ax.set_title("Pairwise Overlap Between Anomaly Detection Methods", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Summary visualization: which depths are flagged by each method
well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

fig, axes = plt.subplots(1, len(methods) + 1, figsize=(22, 10), sharey=True)
fig.suptitle(f"Method Comparison — {well_name}", fontsize=14, y=1.01)

# First panel: GR log for reference
axes[0].plot(wdf["GR"], wdf["MD"], color="green", linewidth=0.5)
axes[0].set_xlabel("GR", fontsize=11)
axes[0].set_title("GR (reference)", fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylabel("Measured Depth (MD)", fontsize=11)
axes[0].invert_yaxis()

# One panel per method showing its flags as coloured depth strips
method_colors = ["steelblue", "coral", "forestgreen", "darkorange", "mediumorchid"]
for i, (name, col, mcolor) in enumerate(zip(method_names, methods, method_colors)):
    ax = axes[i + 1]
    # Plot a binary track
    ax.fill_betweenx(wdf["MD"], 0, wdf[col], color=mcolor, alpha=0.7, step="mid")
    ax.set_xlim(-0.1, 1.1)
    ax.set_xlabel("Flag", fontsize=10)
    ax.set_title(name, fontsize=10)
    ax.set_xticks([0, 1])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Count votes: how many of the five methods flag each row
df_clean["votes"] = df_clean[methods].sum(axis=1)
df_clean["consensus_flag"] = (df_clean["votes"] >= 2).astype(int)

print("=== Votes per Row (a review order, not a verdict) ===")
print(f"Total rows: {len(df_clean)}")
for v in range(6):
    n = (df_clean["votes"] == v).sum()
    print(f"  Flagged by {v} methods: {n} ({100*n/len(df_clean):.2f}%)")
print(f"\nFlagged by 2 or more methods: {df_clean['consensus_flag'].sum()} "
      f"({100*df_clean['consensus_flag'].sum()/len(df_clean):.2f}%)")

# Visualize the consensus on the well log
well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

fig, axes = plt.subplots(1, 5, figsize=(20, 10), sharey=True)
fig.suptitle(f"Rows Flagged by 2 or More Methods — {well_name}", fontsize=14, y=1.01)

curves = ["GR", "RHOB", "NPHI", "DT"]
colors_curve = ["green", "red", "blue", "purple"]

for i, (curve, color) in enumerate(zip(curves, colors_curve)):
    axes[i].plot(wdf[curve], wdf["MD"], color=color, linewidth=0.5, alpha=0.7)
    flagged = wdf[wdf["consensus_flag"] == 1]
    axes[i].scatter(flagged[curve], flagged["MD"], c=flagged["votes"],
                    cmap="Reds", s=10, zorder=5, vmin=2, vmax=5)
    axes[i].set_xlabel(curve, fontsize=11)
    axes[i].set_title(curve, fontsize=12)
    axes[i].grid(True, alpha=0.3)

# Vote count track
axes[4].fill_betweenx(wdf["MD"], 0, wdf["votes"], color="crimson", alpha=0.5, step="mid")
axes[4].axvline(x=2, color="black", linestyle="--", linewidth=1, label="2 votes")
axes[4].set_xlabel("Vote Count", fontsize=11)
axes[4].set_title("Votes", fontsize=12)
axes[4].set_xlim(-0.1, 5.5)
axes[4].grid(True, alpha=0.3)
axes[4].legend(loc="lower right")

axes[0].set_ylabel("Measured Depth (MD)", fontsize=11)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# A review order from the three learned detectors that does not depend on their thresholds:
# convert each score (already higher = more unusual) to a 0-1 percentile rank, then average.
score_cols = ["iso_forest_score", "lof_score", "ocsvm_score"]
review_capacity = 50   # e.g. the number of rows a petrophysicist can review in a day

pct_ranks = df_clean[score_cols].rank(pct=True)     # same as scipy.stats.rankdata(v) / len(v)
df_clean["review_rank"] = pct_ranks.mean(axis=1)    # a review order, not a probability of a fault

review_list = df_clean.sort_values("review_rank", ascending=False).head(review_capacity)
print(f"Top {review_capacity} rows for review, by well:")
print(review_list["wellName"].value_counts().to_string())
print(f"\nVotes among these rows (number of methods that flag them):")
print(review_list["votes"].value_counts().sort_index().to_string())
display(review_list[["wellName", "MD", "GR", "RHOB", "NPHI", "DT", "CALI", "votes", "review_rank"]]
        .head(10).round(3))

### What this comparison can and cannot tell you

- **No precision, recall or accuracy.** The file has no labels, so none of these can be computed for any method. Overlap and votes measure agreement; methods that share assumptions (the z-score and IQR rules) agree for the same reasons and can be wrong together.
- **Stability is not correctness.** The Jaccard overlaps in Part 4 show whether a ranking is reproducible, not whether it is right.
- **Evidence comes from review.** Freeze the detector, split its ranking into bands, and let a petrophysicist review a fixed number of rows from *every* band, unflagged rows included, without seeing the scores. Those labels estimate the share of real problems in each band and how many lie below the threshold.
- **Your alert policy (Lab 2 deliverable, half a page):** what is scored (rows, features, wells); the score and its direction; the threshold and why (review capacity, cost or reviewed intervals, not "contamination = 5%"); rules such as merging adjacent flagged rows into one interval and a minimum interval thickness; who reviews, within what time; what evidence exists and what does not (no labels yet); when the policy is reviewed.

---
## Summary and Key Takeaways

In this exercise we applied five anomaly detection methods to well log data:

1. **Z-score** — simple univariate thresholding; fast but misses multivariate anomalies, and the mean and SD are pulled by the outliers themselves.
2. **IQR** — quartile-based fences, less affected by extreme values; still univariate, and on a mixture of rock types it flags the minority rock type (here, shale-rich intervals).
3. **Isolation Forest** — efficient tree-based method that handles multivariate patterns well.
4. **Local Outlier Factor** — density-based method sensitive to local neighbourhood structure and to `n_neighbors`.
5. **One-Class SVM** — boundary-based method; works well when normal data is compact.

**Key observations:**
- Record what cleaning removed before interpreting any flag: here two of the five wells were dropped, so every result describes three wells.
- A score is not an alert. Every score here is oriented higher = more unusual; the flag is a separate threshold decision.
- `contamination` and `nu` are alert budgets we chose: they fix how many rows are flagged, not how many are bad. With the budget fixed, LOF's `n_neighbors` changed *which* rows were flagged.
- No single method is universally best. Each method has different assumptions about what constitutes an anomaly, and agreement between methods gives a review order, not validation.
- Without labels we cannot report precision or recall. A defensible threshold comes from review capacity or reviewed intervals; for new wells it is chosen on reference or validation data and frozen before use.
- Domain knowledge is essential: a caliper washout (CALI >> BS) is a known cause of unreliable density/neutron readings, and this physics-based check should complement statistical methods.

**Ideas for further exploration:**
- Add a physics-based check: flag depths where CALI significantly exceeds BS (borehole washout).
- Keep all five wells by dropping DT (and the constant BS), or analyse each well separately, and compare the flags.
- Novelty mode: fit the scaler and a detector on two wells, set the threshold from a high quantile of their scores, then score the third well.
- Use DBSCAN for density-based clustering and anomaly detection.
- Apply autoencoders (neural network) for reconstruction-error-based anomaly detection.
- Evaluate the ranking against reviewed intervals: a blind review of flagged and unflagged rows, sampled from every score band.